[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/08_agents_tools_mcp/31_agent_architectures.ipynb)

# 📓 Notebook 31 — Agent Architectures: Loops, Planning & Reflection

> **Module:** Agents, Tools & MCP · **Estimated time:** 70–90 min · **Difficulty:** Intermediate → Advanced

Notebook 24 introduced the **call → execute → return** loop and a small tool-using agent. This module goes professional. Here we study the *architectures* that turn that bare loop into something reliable: **ReAct** (reason + act), **planning** (decompose first, act second), **reflection** (critique and retry), and **memory** (what the agent carries between steps).

Everything runs **100% offline**. A real LLM is the "brain" that decides the next step; to stay reproducible we plug in a small deterministic `policy()` stand-in that makes the *same kind* of decisions a model would. Every place a real provider drops in is marked — swapping it is one line.

> 🎯 **Our running example for the whole module.** Imagine you're building the *brains* of an **AI support copilot** for a SaaS team. It needs to answer questions about support performance — *"what's chat's CSAT?"*, *"take phone's score and multiply by 3"*, *"which channel is doing worst?"* — by looking numbers up and doing arithmetic. We start it as a single agent here; by Notebook 34 it grows into a full multi-agent, MCP-backed assistant. Same scenario, four notebooks — so every pattern you learn lands on the same concrete copilot.

> 🧭 **Mental model to carry the whole way: an agent is an LLM sitting inside a `while`-loop with a memory and a budget.** The clever model isn't the architecture — the *loop around it* is. Hold that one picture and every pattern below (ReAct, planning, reflection, memory) is just a different thing you do *inside* that same loop. We'll keep coming back to it.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Explain the **agent control loop** and why every agent needs a *budget*.
2. Implement the **ReAct** pattern: interleaved Thought → Action → Observation.
3. Add a **planner** that decomposes a task before acting.
4. Add a **reflection** step that critiques an answer and retries on failure.
5. Give an agent **memory** (scratchpad + episodic) and know when each matters.
6. Name the common **failure modes** (loops, hallucinated tools, runaway cost) and the guardrail for each.

## ✅ Prerequisites

Notebook 24 (tool calling & small agents). Functions and dictionaries (NB 4–5). A real LLM is *not* required — everything here runs offline.

## 1. The agent control loop in one picture

An **agent** is a program that lets a model decide *what to do next* in a loop, instead of answering in a single shot:

```
            ┌─────────────────────────────────────────────┐
            │                                             │
   goal ──▶ │  BRAIN (LLM)  ──▶  action: call a tool?     │
            │      ▲                     │                │
            │      │                     ▼                │
            │  observation  ◀──  TOOL runs in YOUR code   │
            │                                             │
            └──────────────── repeat until done ──────────┘
                         (bounded by a step budget)
```

Three rules that never change, no matter how fancy the architecture:

1. **The brain only *decides*. Your code *executes*.** The model never runs anything itself.
2. **Every loop needs a budget.** Max steps / max tool calls — or one bad decision hangs forever.
3. **Every step is logged.** The trace is the only way to debug a misbehaving agent.

## 2. Setup — a tiny world the agent can act in

Before our copilot can *decide* anything, it needs something to act on. We give it the two things any support assistant needs: a way to **look up a number** and a way to **do arithmetic on it**. Concretely: a small in-memory dataset of channel-level CSAT (customer-satisfaction, 1–5) and a calculator. Both are ordinary Python functions — that's all a "tool" ever is.

In [1]:
import json, re

# --- A tiny dataset the agent can look things up in -------------------------
CSAT = {  # channel -> average customer-satisfaction score (1–5)
    "chat":  4.1,
    "email": 3.6,
    "phone": 4.4,
    "social": 3.2,
}

# --- Tools: plain functions that return JSON-serialisable results ------------
def lookup_csat(channel: str) -> dict:
    """Return the average CSAT for a support channel."""
    channel = channel.strip().lower()
    if channel not in CSAT:
        return {"error": f"unknown channel '{channel}'", "known": list(CSAT)}
    return {"channel": channel, "csat": CSAT[channel]}

def calculator(expression: str) -> dict:
    """Evaluate a simple arithmetic expression, e.g. '4.1 * 3'."""
    if not re.fullmatch(r"[0-9.+\-*/() ]+", expression):
        return {"error": "expression contains illegal characters"}
    try:
        return {"result": round(eval(expression, {"__builtins__": {}}), 4)}  # noqa: S307
    except Exception as e:  # noqa: BLE001
        return {"error": str(e)}

TOOLS = {"lookup_csat": lookup_csat, "calculator": calculator}
print("tools available:", list(TOOLS))
print("sample:", lookup_csat("chat"), calculator("4.1 * 3"))

tools available: ['lookup_csat', 'calculator']
sample: {'channel': 'chat', 'csat': 4.1} {'result': 12.3}


## 3. The "brain": a deterministic policy stand-in

A real agent calls an LLM here. The model reads the goal + the trace so far and emits **one** of:

- `{"action": "<tool>", "args": {...}}` — call a tool, or
- `{"final": "<answer>"}` — stop and answer.

So we can run offline and get the *same* result every time, we implement a small rule-based `policy()` that returns exactly that shape. **This is the only piece a real provider replaces** — the loop around it is unchanged.

In [2]:
CHANNELS = "|".join(CSAT)

def policy(goal: str, scratchpad: list[dict]) -> dict:
    """Decide the next step (stand-in for an LLM call).

    Returns either {"thought","action","args"} or {"thought","final"}.
    A real model would *read* the same (goal, scratchpad) and emit the same shape.
    """
    text = goal.lower()
    seen_tools = {step["action"] for step in scratchpad if "action" in step}

    # 1) Need a CSAT number we don't have yet?
    m = re.search(rf"\b({CHANNELS})\b", text)
    if m and "lookup_csat" not in seen_tools:
        return {"thought": f"I need {m.group(1)}'s CSAT score first.",
                "action": "lookup_csat", "args": {"channel": m.group(1)}}

    # 2) Arithmetic asked for, and we have the inputs?
    if any(w in text for w in ("times", "multiply", "*", "sum", "plus", "+")) \
            and "calculator" not in seen_tools:
        csat = next((o["csat"] for o in (s.get("observation", {}) for s in scratchpad)
                     if isinstance(o, dict) and "csat" in o), None)
        if csat is not None and (mult := re.search(r"(\d+(?:\.\d+)?)", text)):
            expr = f"{csat} * {mult.group(1)}"
            return {"thought": f"Now multiply: {expr}.",
                    "action": "calculator", "args": {"expression": expr}}

    # 3) Otherwise, answer from what we've observed.
    facts = [s["observation"] for s in scratchpad if "observation" in s]
    return {"thought": "I have enough to answer.", "final": _summarise(goal, facts)}

def _summarise(goal: str, facts: list) -> str:
    if not facts:
        return "I could not find the information needed."
    last = facts[-1]
    if isinstance(last, dict) and "result" in last:
        return f"The answer is {last['result']}."
    if isinstance(last, dict) and "csat" in last:
        return f"{last['channel'].title()} has an average CSAT of {last['csat']}."
    return f"Result: {last}"

print(policy("What is chat's CSAT?", []))

{'thought': "I need chat's CSAT score first.", 'action': 'lookup_csat', 'args': {'channel': 'chat'}}


## 4. ReAct: interleave Thought → Action → Observation

**ReAct** ("Reason + Act") is the workhorse agent pattern. Each iteration the brain writes a short *thought*, picks an *action*, your code runs it, and the *observation* is fed back. The loop ends when the brain emits a `final`, or when the **step budget** runs out.

Notice how small the loop is — the architecture is in the *policy*, not the plumbing.

### 🔬 What actually happens — an agent is a state machine, not a smart function

Notebook 24 taught you the *one-shot* mechanic: the model emits a tool call, your code runs it, you hand the result back, the model replies. That is a **single turn**. An *agent architecture* is what you get when you wrap that turn in a **loop with state** and let the model decide — fresh, every iteration — whether to keep going.

The architecture isn't the model. It's four pieces of **state** plus a **stop rule**:

| State | What it holds | Why it exists |
|---|---|---|
| **`goal`** | the task, fixed for the whole run | the model re-reads it every step to stay on target |
| **`scratchpad`** | the list of past *(thought, action, observation)* steps | the model is **stateless** — without this it forgets what it already did |
| **`step`** | a counter, `1, 2, 3, …` | needed to enforce the budget |
| **`max_steps`** | the budget (a constant) | the **stop rule** — guarantees the loop *terminates* |

> 🧭 **Mental model, made precise.** This is the *"LLM in a `while`-loop with a memory and a budget"* picture from the intro, spelled out: the **memory** is the `scratchpad`, the **budget** is `max_steps`, and the **loop** is the `for`/`while` around the one model call. The architecture is the **loop + state + stop rule** — *not* the model. Swap the model for a smarter one and the loop is byte-for-byte identical; that's why the same `react_agent` plumbing below drives the offline `policy()` and a real provider alike. For our support copilot, *"what's chat's CSAT × 3?"* is just two trips around this loop: look up the number, then multiply.


### The control loop as a state machine

Read each iteration as a transition between states. The model only ever picks **one** of two outgoing edges: *ACT* (call a tool) or *STOP* (final answer). Your code does everything else.

```text
                         ┌─────────────────────────────────────────────┐
   goal ───────────────▶ │  state = (goal, scratchpad=[], step=0)       │
                         └───────────────────┬─────────────────────────┘
                                             │
              ┌──────────────────────────────▼──────────────────────────────┐
              │  step += 1                                                    │
   ┌──────────┤  IF step > max_steps ──────────────▶ STOP: "(budget exhausted)"│ ◀── stop rule #1
   │          │  decision = model(goal, scratchpad)   # the ONLY model call    │
   │          └──────────────────────────────┬──────────────────────────────┘
   │                                          │
   │              ┌───────────────────────────┴───────────────────────────┐
   │              │                                                         │
   │          decision is ACT                                      decision is STOP ◀── stop rule #2
   │     {"action": tool, "args": …}                            {"final": "answer"}
   │              │                                                         │
   │   observation = tool(**args)        # YOUR code runs it                │
   │   scratchpad.append(                                                   ▼
   │       {thought, action, args, observation})                  return answer ✅
   │              │
   └──────────────┘   loop back — model now sees one more observation
```

Two — and *only* two — ways out of the loop, the two **stop conditions**:

1. **The model says stop** (`final`) — the normal, happy exit.
2. **The budget runs out** (`step > max_steps`) — the safety exit that guarantees the program can't hang.

Every trip around the loop the scratchpad grows by one entry, so the model's *view of the world* changes — which is why a deterministic model can still produce a different decision on step 2 than it did on step 1.


### Fixed pipeline vs. agent — who decides the next step?

This is *the* distinction that makes something an "agent architecture". In a **fixed pipeline** you, the programmer, wrote the order of steps in advance. In an **agent**, the *model* chooses the next step at runtime, looking at the scratchpad so far.

```text
FIXED PIPELINE  (control flow is in YOUR code — written ahead of time)
   ┌────────┐   ┌────────┐   ┌────────┐
   │ step 1 │──▶│ step 2 │──▶│ step 3 │──▶ done     ← you decided this order
   └────────┘   └────────┘   └────────┘                the model can't change it

AGENT           (control flow is DECIDED EACH ITERATION by the model)
        ┌───────────────── model looks at scratchpad ─────────────────┐
        ▼                                                             │
   "what next?" ──▶ tool A ──▶ "what next?" ──▶ tool B ──▶ "what next?" ┘──▶ final
        ▲              the NUMBER and ORDER of steps is unknown up front
```

| | Fixed pipeline | Agent |
|---|---|---|
| **Who picks the next step** | the programmer, in advance | the model, at runtime |
| **Number of steps** | known before running | unknown — depends on the task |
| **Order of steps** | hard-coded | emergent from the scratchpad |
| **Can it loop forever?** | no (no loop) | **yes** — hence the budget |
| **Good for** | known, repeatable workflows | open-ended tasks where the path varies |

> ⚠️ **This freedom is exactly why a budget is non-negotiable.** A fixed pipeline *can't* loop forever because there's no loop. An agent can: if the model keeps deciding "call the same tool again", nothing stops it but the `max_steps` guard. *Every* agent loop you write needs a stop rule, on day one — not as a later optimisation.


In [3]:
# 🧪 PROOF — a self-contained mock ReAct loop: scratchpad + step budget + stop rule.
# Stdlib only, fully deterministic, no API. This is the skeleton of EVERY agent below.

# --- a tiny tool the agent can call ----------------------------------------
PRICES = {"widget": 4.0, "gadget": 7.5}
def get_price(item: str) -> dict:
    item = item.strip().lower()
    return {"item": item, "price": PRICES[item]} if item in PRICES else {"error": f"unknown item {item!r}"}

# --- the "brain": deterministic stand-in for an LLM ------------------------
# It reads (goal, scratchpad) and emits ONE of:
#   {"thought","action","args"}  -> ACT          (call a tool)
#   {"thought","final"}          -> STOP          (answer)
def fake_llm(goal: str, scratchpad: list[dict]) -> dict:
    seen = {s["action"] for s in scratchpad if "action" in s}     # tools already used
    if "get_price" not in seen:                                   # step 1: fetch the price
        return {"thought": "I don't know the price yet — look it up.",
                "action": "get_price", "args": {"item": "widget"}}
    price = scratchpad[-1]["observation"]["price"]                # step 2: reason over it
    return {"thought": f"I have the price ({price}); 10 units cost {price * 10}.",
            "final": f"10 widgets cost {price * 10}."}

# --- the loop: this IS the architecture (state + loop + stop rule) ----------
def mock_react(goal: str, max_steps: int = 4) -> str:
    scratchpad: list[dict] = []          # STATE: memory of past steps (model is stateless)
    print(f"GOAL: {goal}\nbudget: max_steps={max_steps}\n" + "-" * 52)
    for step in range(1, max_steps + 1):                         # STOP RULE #1: the budget
        decision = fake_llm(goal, scratchpad)                    # the ONE model call per turn
        print(f"[step {step}] Thought: {decision['thought']}")
        if "final" in decision:                                  # STOP RULE #2: model says stop
            print(f"[step {step}] Final Answer: {decision['final']}")
            return decision["final"]
        name, args = decision["action"], decision["args"]
        observation = {"get_price": get_price}[name](**args)     # YOUR code runs the tool
        print(f"[step {step}] Action: {name}({args})")
        print(f"[step {step}] Observation: {observation}")
        scratchpad.append({"thought": decision["thought"], "action": name,
                           "args": args, "observation": observation})  # grow the scratchpad
    return "(budget exhausted)"                                  # safety exit — never hangs

answer = mock_react("How much do 10 widgets cost?")
print("-" * 52)
print("RETURNED:", answer)


GOAL: How much do 10 widgets cost?
budget: max_steps=4
----------------------------------------------------
[step 1] Thought: I don't know the price yet — look it up.
[step 1] Action: get_price({'item': 'widget'})
[step 1] Observation: {'item': 'widget', 'price': 4.0}
[step 2] Thought: I have the price (4.0); 10 units cost 40.0.
[step 2] Final Answer: 10 widgets cost 40.0.
----------------------------------------------------
RETURNED: 10 widgets cost 40.0.


### Reading the trace, and proving the budget actually saves you

The print-out above **is** the ReAct trace — the canonical `Thought → Action → Observation`, repeated, until a `Final Answer`:

```text
Thought:      "I don't know the price yet — look it up."   ← reason
Action:       get_price({'item': 'widget'})                ← act
Observation:  {'item': 'widget', 'price': 4.0}             ← result fed back into the scratchpad
Thought:      "I have the price (4.0); 10 units cost 40.0" ← reason again, now with the fact
Final Answer: "10 widgets cost 40.0."                       ← stop
```

Each `Observation` is appended to the `scratchpad`, so the *next* `Thought` is computed from a richer view — that loop-of-growing-memory is the whole trick. Now watch what the **budget** does when a (broken) brain *never* says `final`: instead of hanging forever, the loop exits cleanly. This is stop-rule #1 earning its keep.


In [4]:
# 🧪 PROOF — the budget guarantees termination even for a brain that loops forever.
def broken_llm(goal: str, scratchpad: list[dict]) -> dict:
    # A buggy "model" that ALWAYS asks for the same tool and NEVER answers.
    return {"thought": "let me check the price (again)...",
            "action": "get_price", "args": {"item": "widget"}}

def mock_react_with(brain, goal: str, max_steps: int = 3) -> str:
    scratchpad: list[dict] = []
    for step in range(1, max_steps + 1):           # without this bound -> infinite loop
        decision = brain(goal, scratchpad)
        if "final" in decision:
            return decision["final"]
        obs = get_price(**decision["args"])
        scratchpad.append({"action": decision["action"], "args": decision["args"], "observation": obs})
        print(f"[step {step}] {decision['action']}({decision['args']}) -> {obs}")
    return "(budget exhausted — stopped a runaway agent)"

print("RESULT:", mock_react_with(broken_llm, "price of a widget?", max_steps=3))
print("\nThe agent thrashed 3 times, then the stop rule cut it off — no hang, no runaway cost.")


[step 1] get_price({'item': 'widget'}) -> {'item': 'widget', 'price': 4.0}
[step 2] get_price({'item': 'widget'}) -> {'item': 'widget', 'price': 4.0}
[step 3] get_price({'item': 'widget'}) -> {'item': 'widget', 'price': 4.0}
RESULT: (budget exhausted — stopped a runaway agent)

The agent thrashed 3 times, then the stop rule cut it off — no hang, no runaway cost.


In [5]:
def react_agent(goal: str, tools: dict, policy_fn, max_steps: int = 6, verbose: bool = True):
    """A ReAct loop: think, act, observe — until 'final' or budget exhausted."""
    scratchpad: list[dict] = []          # the agent's working memory
    trace: list[dict] = []
    for step in range(1, max_steps + 1):
        decision = policy_fn(goal, scratchpad)
        if "final" in decision:
            if verbose: print(f"[{step}] 💭 {decision['thought']}\n    ✅ {decision['final']}")
            trace.append({"step": step, **decision})
            return {"answer": decision["final"], "steps": step, "trace": trace}

        name, args = decision["action"], decision.get("args", {})
        if name not in tools:            # guardrail: hallucinated tool
            obs = {"error": f"no such tool '{name}'"}
        else:
            obs = tools[name](**args)
        scratchpad.append({"action": name, "args": args, "observation": obs})
        trace.append({"step": step, "thought": decision["thought"],
                      "action": name, "args": args, "observation": obs})
        if verbose:
            print(f"[{step}] 💭 {decision['thought']}")
            print(f"    🔧 {name}({args}) -> {obs}")
    return {"answer": "(budget exhausted)", "steps": max_steps, "trace": trace}

out = react_agent("What is the average CSAT for chat?", TOOLS, policy)
print("\nFINAL:", out["answer"], "in", out["steps"], "steps")

[1] 💭 I need chat's CSAT score first.
    🔧 lookup_csat({'channel': 'chat'}) -> {'channel': 'chat', 'csat': 4.1}
[2] 💭 I have enough to answer.
    ✅ Chat has an average CSAT of 4.1.

FINAL: Chat has an average CSAT of 4.1. in 2 steps


A **multi-step** question forces the loop to chain tools — look up a number, then compute with it — with no change to the agent code:

In [6]:
out = react_agent("Take chat's CSAT and multiply it by 3", TOOLS, policy)
print("\nFINAL:", out["answer"])

[1] 💭 I need chat's CSAT score first.
    🔧 lookup_csat({'channel': 'chat'}) -> {'channel': 'chat', 'csat': 4.1}
[2] 💭 Now multiply: 4.1 * 3.
    🔧 calculator({'expression': '4.1 * 3'}) -> {'result': 12.3}
[3] 💭 I have enough to answer.
    ✅ The answer is 12.3.

FINAL: The answer is 12.3.


---

### ✋ Quick exercise (~2 min) — Predict the ReAct trace

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Your support copilot is asked: *"Take social's CSAT and multiply by 2."* Using only the `policy` rules and `react_agent` loop above, **predict before running**: how many steps does the loop take, and what is the final answer string?

```python
goal = "Take social's CSAT and multiply by 2"   # social's CSAT is in the CSAT dict
```

In [7]:
# ✍️ Your turn 👇
# First write down your prediction (steps + answer), THEN run to check:
# out = react_agent("Take social's CSAT and multiply by 2", TOOLS, policy, verbose=False)
# print(out["steps"], out["answer"])


<details>
<summary>✅ <b>Solution</b></summary>

```python
out = react_agent("Take social's CSAT and multiply by 2", TOOLS, policy, verbose=False)
print(out["steps"], "steps ->", out["answer"])
# 3 steps -> The answer is 6.4.
```

The loop takes **3 steps**: step 1 `lookup_csat(social)` (→ 3.2), step 2 `calculator("3.2 * 2")` (→ 6.4), step 3 a `final`. Chaining a lookup into arithmetic always needs the two tool steps plus the answer step.
</details>

## 5. Planning: decompose *before* acting

ReAct decides one step at a time — great for open-ended tasks, but it can wander. **Plan-and-execute** flips the order: the brain first writes a *plan* (an ordered list of sub-goals), then executes each. Plans make the agent's intent inspectable *before* it spends a single tool call, and they're easy to validate or approve.

In [8]:
def planner(goal: str) -> list[str]:
    """Stand-in for an LLM that decomposes a goal into ordered sub-goals."""
    g = goal.lower()
    plan = []
    for ch in CSAT:
        if ch in g:
            plan.append(f"Look up CSAT for {ch}")
    if any(w in g for w in ("compare", "highest", "best", "lowest", "worst")):
        plan.append("Compare the looked-up scores")
    return plan or ["Answer directly"]

def plan_and_execute(goal: str, tools: dict, verbose: bool = True):
    plan = planner(goal)
    if verbose: print("📝 PLAN:"); [print(f"   {i}. {s}") for i, s in enumerate(plan, 1)]
    results = {}
    for ch in CSAT:
        if any(ch in s.lower() for s in plan):
            results[ch] = tools["lookup_csat"](channel=ch)["csat"]
    answer = (f"Highest CSAT: {max(results, key=results.get)} "
              f"({max(results.values())})") if results else "Nothing to compare."
    if verbose: print("✅", answer)
    return {"plan": plan, "results": results, "answer": answer}

_ = plan_and_execute("Compare chat and phone and tell me the highest CSAT", TOOLS)

📝 PLAN:
   1. Look up CSAT for chat
   2. Look up CSAT for phone
   3. Compare the looked-up scores
✅ Highest CSAT: phone (4.4)


---

### ✋ Quick exercise (~2 min) — Make the planner emit a 3-step plan

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using only the `planner` function above, write a single `goal` string that makes `planner(goal)` return **exactly three** sub-goals: two channel look-ups plus one compare step. Predict the list it returns, then run `print(planner(goal))` to confirm.

In [9]:
# ✍️ Your turn 👇
# goal = "..."          # craft a goal that yields a 3-item plan
# print(planner(goal))


<details>
<summary>✅ <b>Solution</b></summary>

```python
goal = "Compare chat and phone CSAT"
print(planner(goal))
# ['Look up CSAT for chat', 'Look up CSAT for phone', 'Compare the looked-up scores']
```

`planner` appends one *"Look up CSAT for …"* sub-goal per channel name it finds in the text, then adds *"Compare the looked-up scores"* because it sees a comparison word ("compare"). Two channel names + one comparison keyword = a 3-item plan.
</details>

## 6. Reflection: critique, then retry

A single pass can be wrong. **Reflection** adds a critic step: after producing an answer, the agent (or a second "critic" brain) checks it against the goal and either accepts it or sends feedback for another attempt. This is the cheapest reliability upgrade you can add.

In [10]:
def critic(goal: str, answer: str) -> dict:
    """Stand-in critic: is the answer acceptable? If not, why?"""
    if "could not" in answer.lower() or answer.endswith("None."):
        return {"ok": False, "feedback": "No concrete value produced — try a tool first."}
    if not re.search(r"\d", answer):
        return {"ok": False, "feedback": "Answer has no number; the goal expects one."}
    return {"ok": True, "feedback": "Looks grounded."}

def reflexive_agent(goal: str, tools: dict, max_attempts: int = 2):
    for attempt in range(1, max_attempts + 1):
        out = react_agent(goal, tools, policy, verbose=False)
        verdict = critic(goal, out["answer"])
        print(f"Attempt {attempt}: {out['answer']!r} → critic: {verdict['feedback']}")
        if verdict["ok"]:
            return out["answer"]
    return out["answer"] + "  (accepted after retries)"

print("\nRESULT:", reflexive_agent("What is phone's CSAT?", TOOLS))

Attempt 1: 'Phone has an average CSAT of 4.4.' → critic: Looks grounded.

RESULT: Phone has an average CSAT of 4.4.


---

### ✋ Quick exercise (~2 min) — Will the critic accept it?

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using the `critic` function above, **predict** the `ok` verdict for each of these two candidate copilot answers, then call `critic` to check:

```python
"Phone has an average CSAT of 4.4."
"Sorry, no data available."
```

In [11]:
# ✍️ Your turn 👇
# Predict each verdict first, then run:
# print(critic("phone csat", "Phone has an average CSAT of 4.4."))
# print(critic("phone csat", "Sorry, no data available."))


<details>
<summary>✅ <b>Solution</b></summary>

```python
print(critic("phone csat", "Phone has an average CSAT of 4.4."))  # {'ok': True,  ...}
print(critic("phone csat", "Sorry, no data available."))          # {'ok': False, ...}
```

The first answer contains a digit, so `critic` returns `ok=True` ("Looks grounded."). The second has **no number**, so the `re.search(r"\d", answer)` check fails and `critic` rejects it — which is what would trigger another attempt inside `reflexive_agent`.
</details>

## 7. Memory: scratchpad vs episodic

Two kinds of memory, often confused:

| Memory | Lives for | Holds | In this notebook |
|---|---|---|---|
| **Scratchpad** (working) | one task | the current Thought/Action/Observation trail | the `scratchpad` list |
| **Episodic** (long-term) | across tasks | past goals & outcomes, to reuse | the `EPISODES` store below |

Episodic memory is what lets an agent say *"I answered this yesterday"* — a simple key→answer cache is often enough to start.

In [12]:
EPISODES: dict[str, str] = {}   # goal -> answer, persists across calls

def agent_with_memory(goal: str, tools: dict):
    if goal in EPISODES:
        print("🧠 episodic hit — reusing prior answer")
        return EPISODES[goal]
    answer = react_agent(goal, tools, policy, verbose=False)["answer"]
    EPISODES[goal] = answer
    return answer

print(agent_with_memory("What is the average CSAT for email?", TOOLS))
print(agent_with_memory("What is the average CSAT for email?", TOOLS))  # served from memory
print("episodes remembered:", len(EPISODES))

Email has an average CSAT of 3.6.
🧠 episodic hit — reusing prior answer
Email has an average CSAT of 3.6.
episodes remembered: 1


---

### ✋ Quick exercise (~2 min) — Predict the episodic cache hits

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Using `agent_with_memory` and the `EPISODES` cache above, you call the copilot three times for chat's CSAT — twice with identical wording, once with different capitalisation:

```python
"What is chat's CSAT?"
"What is chat's CSAT?"
"what is chat's csat?"
```

**Predict:** how many of the three calls are served from episodic memory (print the "episodic hit" message)? Then run them to check.

In [13]:
# ✍️ Your turn 👇
# Predict how many of these three calls print the "episodic hit" message, then run:
# agent_with_memory("What is chat's CSAT?", TOOLS)
# agent_with_memory("What is chat's CSAT?", TOOLS)
# agent_with_memory("what is chat's csat?", TOOLS)


<details>
<summary>✅ <b>Solution</b></summary>

```python
agent_with_memory("What is chat's CSAT?", TOOLS)   # miss — computes & stores
agent_with_memory("What is chat's CSAT?", TOOLS)   # hit  — exact key match 🧠
agent_with_memory("what is chat's csat?", TOOLS)   # miss — different string, new key
```

Only **1 of 3** is served from memory. `EPISODES` is keyed on the *exact* goal string, so the second call hits, but the third differs in capitalisation and punctuation, producing a new key and a recompute. (Stretch D later fixes this by normalising the key.)
</details>

## 8. Failure modes & the guardrail for each

Agents fail in a small number of predictable ways. Memorise this table — it is most of "production agent engineering":

| Failure | Symptom | Guardrail |
|---|---|---|
| **Infinite loop** | same tool, same args, forever | `max_steps` budget (we set it) |
| **Hallucinated tool** | calls a tool that doesn't exist | check `name in tools`, return an error obs |
| **Runaway cost** | dozens of model calls per question | cap tool calls *and* tokens; alert on spend |
| **Bad arguments** | tool raises / returns garbage | validate args (next notebook) |
| **Silent wrong answer** | confident but ungrounded | reflection / critic step |
| **Wandering** | explores irrelevant tools | plan-first, or constrain the toolset |

The one-line summary: **budget every loop, validate every input, log every step, and critique the output.**

## 9. Going live — the real-provider sketch

Offline we used `policy()`. With a real provider you delete it and let the model decide, passing your tools as schemas. The loop body is identical.

In [14]:
# Reference only — uncomment with a real key + `pip install anthropic`.
#
# ⚠️ NOTE: the course's `llm_providers.py` wrappers are TEXT-ONLY — their .chat()
# returns just the text and discards tool_use blocks, so they CANNOT drive tool
# calling. With a real provider, use the raw SDK instead: pass your schemas to
# `client.messages.create(..., tools=...)` and read the `tool_use` content
# blocks from the response.
#
# import anthropic
# client = anthropic.Anthropic()                     # reads ANTHROPIC_API_KEY
#
# TOOL_SCHEMAS = [{
#     "name": "lookup_csat",
#     "description": "Average customer-satisfaction (1-5) for a support channel.",
#     "input_schema": {"type": "object",
#                      "properties": {"channel": {"type": "string"}},
#                      "required": ["channel"]},
# }]
#
# def llm_policy(goal, scratchpad):
#     msgs = [{"role": "user", "content": goal + "\n\nTrace:\n" + json.dumps(scratchpad)}]
#     r = client.messages.create(model="claude-haiku-4-5", max_tokens=1024,
#                                messages=msgs, tools=TOOL_SCHEMAS)
#     for block in r.content:                        # text and/or tool_use blocks
#         if block.type == "tool_use":
#             return {"action": block.name, "args": block.input}
#     return {"final": "".join(b.text for b in r.content if b.type == "text")}
#
# react_agent(goal, TOOLS, llm_policy)   # SAME loop, real brain.
print("(reference cell — the offline policy above is the runnable version)")

(reference cell — the offline policy above is the runnable version)


## 🧪 Practice exercises

Each ships with a worked solution. Try first, then expand.

### Exercise 1 — ⭐ Add a `list_channels` tool

Add a tool that returns all known channels, register it in `TOOLS`, and call it directly.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def list_channels() -> dict:
    return {"channels": list(CSAT)}

TOOLS["list_channels"] = list_channels
print(list_channels())
```

**Reasoning:** Tools are plain Python callables held in a dict, so adding a capability is a one-line insert into `TOOLS`. Calling the function directly first proves it works before any agent loop ever depends on it.
</details>

### Exercise 2 — ⭐⭐ Pretty-print a trace

Write `show_trace(out)` that prints each step as `step. tool(args) -> observation`.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def show_trace(out: dict) -> None:
    for s in out["trace"]:
        if "action" in s:
            print(f"{s['step']}. {s['action']}({s['args']}) -> {s['observation']}")
        else:
            print(f"{s['step']}. FINAL: {s['final']}")

show_trace(react_agent("chat csat times 2", TOOLS, policy, verbose=False))
```

**Reasoning:** Trace entries are heterogeneous: tool steps carry `action`/`args`/`observation`, while the final step only carries `final`. Branching on key presence handles both shapes without crashing on either.
</details>

### Exercise 3 — ⭐⭐ Lower the budget

Run a multi-step question with `max_steps=1`. Confirm the agent reports `(budget exhausted)` rather than hanging.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
tight = react_agent("Take phone's CSAT and multiply by 5", TOOLS, policy,
                     max_steps=1, verbose=False)
print(tight["answer"], "| steps used:", tight["steps"])
```

**Reasoning:** The `for`-loop budget guarantees termination: after one iteration the loop ends and the agent returns `(budget exhausted)` instead of looping forever. The answer is worse, but the *behaviour* is predictable — exactly what you want in production.
</details>

### Exercise 4 — ⭐⭐ Debug me 🐞

The cell below is meant to count how many steps *used a tool*, but it raises. Read the error, then fix it (expand the solution below when ready).

In [15]:
out = react_agent("compare chat and email", TOOLS, policy, verbose=False)
# 🐞 BUG (INTENTIONALLY ERRORS): not every trace entry has an "action" key.
tool_steps = sum(1 for s in out["trace"] if s["action"] != "final")
print(tool_steps)

KeyError: 'action'

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
# ✅ Fix: guard for the key — final steps have no "action".
out = react_agent("compare chat and email", TOOLS, policy, verbose=False)
tool_steps = sum(1 for s in out["trace"] if "action" in s)
print("tool-using steps:", tool_steps)
```

**Reasoning:** Final steps have no `"action"` key, so indexing it raises `KeyError`. Counting entries where the key *exists* (`"action" in s`) sidesteps the crash and counts exactly the tool-using steps.
</details>

## 🧠 Stretch exercises

### Stretch A — ⭐⭐⭐ A `max_tool_calls` budget

`max_steps` bounds iterations; add a separate cap on *tool executions* (final steps don't count).

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def budgeted_agent(goal, tools, max_tool_calls=2):
    scratch, calls = [], 0
    while True:
        d = policy(goal, scratch)
        if "final" in d:                       # a final answer always returns
            return d["final"]
        if calls >= max_tool_calls:            # the budget limits *tool calls* only
            return f"(stopped: hit {max_tool_calls}-tool-call budget)"
        scratch.append({"action": d["action"], "args": d["args"],
                        "observation": tools[d["action"]](**d["args"])})
        calls += 1

print(budgeted_agent("chat csat times 3", TOOLS, max_tool_calls=2))
print(budgeted_agent("chat csat times 3", TOOLS, max_tool_calls=1))
```

**Reasoning:** Two separate budgets do two separate jobs: `max_steps` bounds loop iterations, while `calls` counts only *tool executions*. A final answer is always allowed through — the cap should stop runaway tool use, not block the agent from finishing.
</details>

### Stretch B — ⭐⭐⭐ Loop detection

Stop early if the agent requests the *same* (tool, args) twice — a classic stuck-loop signal.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def loop_safe_agent(goal, tools, max_steps=8):
    scratch, seen = [], set()
    for _ in range(max_steps):
        d = policy(goal, scratch)
        if "final" in d:
            return d["final"]
        key = (d["action"], json.dumps(d["args"], sort_keys=True))
        if key in seen:
            return f"(stopped: repeated call {d['action']}{d['args']})"
        seen.add(key)
        scratch.append({"action": d["action"], "args": d["args"],
                        "observation": tools[d["action"]](**d["args"])})
    return "(budget exhausted)"

print(loop_safe_agent("What is social's CSAT?", TOOLS))
```

**Reasoning:** Each (tool, args) pair is canonicalised with `json.dumps(..., sort_keys=True)` so identical requests always produce the same key. Seeing the same key twice means the policy is stuck — stopping early converts a silent infinite loop into a visible, debuggable message.
</details>

### Stretch C — ⭐⭐⭐ A two-critic vote

Reliability scales with independent checks. Run two different critics and accept only if **both** pass.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def critic_has_number(goal, ans): return {"ok": bool(re.search(r"\d", ans))}
def critic_not_empty(goal, ans):  return {"ok": len(ans.strip()) > 5}

def doubly_checked(goal, tools):
    ans = react_agent(goal, tools, policy, verbose=False)["answer"]
    votes = [critic_has_number(goal, ans)["ok"], critic_not_empty(goal, ans)["ok"]]
    return ans if all(votes) else ans + "  ⚠️ failed a check"

print(doubly_checked("phone csat", TOOLS))
```

**Reasoning:** The two critics check *independent* properties (contains a number, is non-trivially long), so a bad answer must slip past both to be accepted. Requiring `all(votes)` is the cheapest form of defence in depth — reliability scales with independent checks.
</details>

### Stretch D — ⭐⭐⭐ Episodic memory with normalisation

`agent_with_memory` misses *"what is chat csat?"* vs *"What is chat CSAT?"*. Normalise the key (lowercase, strip punctuation) so paraphrases hit the cache.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def norm_key(goal: str) -> str:
    return re.sub(r"[^a-z0-9 ]", "", goal.lower()).strip()

MEM2: dict[str, str] = {}
def smart_memory_agent(goal, tools):
    k = norm_key(goal)
    if k in MEM2:
        return f"(cached) {MEM2[k]}"
    MEM2[k] = react_agent(goal, tools, policy, verbose=False)["answer"]
    return MEM2[k]

print(smart_memory_agent("What is chat CSAT?", TOOLS))
print(smart_memory_agent("what is chat csat", TOOLS))   # paraphrase → cache hit
```

**Reasoning:** Normalising the key (lowercase, strip punctuation) maps paraphrases like *"What is chat CSAT?"* and *"what is chat csat"* to the same cache entry, so the second call is a hit instead of a recompute. The trade-off: more aggressive normalisation risks collapsing genuinely different goals.
</details>

## 🎁 Bonus mini-project — a self-describing agent

Combine the pieces: an agent that, given any goal, **prints its plan, runs ReAct, then reflects** — and returns a tidy report dict. This is the skeleton you'll grow in the rest of the module.

<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
def reporting_agent(goal: str, tools: dict) -> dict:
    plan = planner(goal)
    out = react_agent(goal, tools, policy, verbose=False)
    verdict = critic(goal, out["answer"])
    report = {"goal": goal, "plan": plan, "answer": out["answer"],
              "steps": out["steps"], "accepted": verdict["ok"]}
    print(json.dumps(report, indent=2))
    return report

_ = reporting_agent("Take email's CSAT and multiply by 10", TOOLS)
```

**Reasoning:** This stitches the module together: `planner` makes intent inspectable before acting, `react_agent` does the bounded work, and `critic` gates the answer. Returning a tidy report dict (instead of just printing) makes the agent observable — the property every production agent needs first.
</details>

## 🧠 Key takeaways

> 🧭 **Back to the loop.** Every pattern in this notebook was the *same* support copilot, the *same* `while`-loop, with one thing changed inside it. **ReAct** changed *what the brain decides each turn*; **planning** changed *when* it decides (all at once, up front); **reflection** added *a second brain that checks the answer*; **memory** changed *what the loop remembers* (this task, or across tasks). The model never changed — and neither did the loop. That's the whole point of an architecture: it's the scaffolding around the brain, and it's *yours* to design. Hold onto that "LLM-in-a-loop-with-memory-and-a-budget" picture — Notebook 32 zooms into the *tools* the loop calls, NB 33 standardises how it reaches them, and NB 34 runs *several* of these loops as a team.

1. An **agent** = a model deciding the next step in a **bounded loop**; your code executes, the model only decides.
2. **ReAct** interleaves Thought → Action → Observation — flexible, but bound it with a step budget.
3. **Planning** decomposes first so intent is inspectable *before* tools run.
4. **Reflection** (a critic step) is the cheapest reliability upgrade — critique, then retry.
5. **Scratchpad** memory is per-task; **episodic** memory persists across tasks.
6. Agents fail in a *small, known* set of ways — budget loops, validate inputs, log steps, critique outputs.
7. The offline `policy()` and a real LLM are interchangeable: **the loop never changes.**

## ✅ Self-assessment

- [ ] Draw the agent control loop and state the three invariant rules
- [ ] Implement a ReAct loop with a `max_steps` budget
- [ ] Add a planner that decomposes a goal before acting
- [ ] Add a critic that rejects ungrounded answers and triggers a retry
- [ ] Distinguish scratchpad from episodic memory and implement each
- [ ] Name a guardrail for every failure mode in the table

## 🚀 Next step

Continue with **Notebook 32 — Designing Robust Tools**, where we make the *tools themselves* production-grade: typed schemas, input validation, structured errors, approval gates, and parallel calls. Same copilot, but now we harden the `lookup_csat` and `calculator` it leans on so a wrong argument can never become a wrong action.